In [2]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

In [3]:
# function to get list of all race id in a season
# series key for now is 'series_1' for cup, 'series_2' for xfinity, 'series_3' for trucks
def get_race_ids(year, series_key):

    race_list_url = f"https://cf.nascar.com/cacher/{year}/race_list_basic.json"
    response = requests.get(race_list_url)
    data = response.json()
    races_df = pd.DataFrame(data[series_key])
    races_df_clean = races_df[['race_id', 'series_id', 'race_season', 'track_id', 'race_name', 'race_type_id', 'actual_laps']]
    return races_df_clean


In [4]:
# function to get track information 
def get_track_info():
    track_info_url = "https://cf.nascar.com/cacher/tracks.json"
    response = requests.get(track_info_url)
    data = response.json()
    df = pd.DataFrame(data['items'])
    df = df[['track_id', 'track_name', 'track_surface', 'track_type']]
    return df

### Getting race data for each race in 2025

In [5]:

year = 2024
series_key = 'series_1'
races = get_race_ids(year, series_key)
races = races[races['race_type_id'] ==1]
races.head()


,race_id,series_id,race_season,track_id,race_name,race_type_id,actual_laps
3,5385,1,2024,105,DAYTONA 500,1,200
4,5389,1,2024,111,Ambetter Health 400,1,260
5,5387,1,2024,42,Pennzoil 400 presented by Jiffy Lube,1,267
6,5388,1,2024,84,Shriners Children’s 500,1,312
7,5392,1,2024,14,Food City 500,1,500


### track data for all tracks

In [6]:
track_df = get_track_info()
track_df.head()


,track_id,track_name,track_surface,track_type
0,4,Darlington Raceway,Asphalt,Intermediate
1,14,Bristol Motor Speedway,Concrete,Short Track
2,22,Martinsville Speedway,Asphalt,Short Track
3,26,Richmond Raceway,Asphalt,Short Track
4,38,Auto Club Speedway,Asphalt,Intermediate


### merging race data with the track data

In [7]:
races = races.merge(track_df, on='track_id', how='left')
races.head()

,race_id,series_id,race_season,track_id,race_name,race_type_id,actual_laps,track_name,track_surface,track_type
0,5385,1,2024,105,DAYTONA 500,1,200,Daytona International Speedway,Asphalt,Superspeedway
1,5389,1,2024,111,Ambetter Health 400,1,260,EchoPark Speedway,Asphalt,Drafting
2,5387,1,2024,42,Pennzoil 400 presented by Jiffy Lube,1,267,Las Vegas Motor Speedway,Asphalt,Intermediate
3,5388,1,2024,84,Shriners Children’s 500,1,312,Phoenix Raceway,Asphalt,Intermediate
4,5392,1,2024,14,Food City 500,1,500,Bristol Motor Speedway,Concrete,Short Track


In [ ]:
# function to get lap data for a single race
def fetch_single_race(year,series_num, race_id):
    race_url = f"https://cf.nascar.com/cacher/{year}/{series_num}/{race_id}/lap-times.json"
    response = requests.get(race_url)
    data = response.json()
    # pull out lap data and create a row for each driver lap time
    df = pd.json_normalize(
        data['laps'],
        record_path='Laps',
        meta=['Number', 'FullName', 'NASCARDriverID']
    )
    # will need to adjust the flag key if more flag states are added in the future
    flag_key = {
    0: 'None',
    1: 'Green',
    2: 'Yellow',
    3: 'Red',
    4: 'White',
    5: 'Checkered',
    8: 'Hot Track',
    9: 'Cold Track'}
    # create a dataframe for flag status and merge it with the lap data
    flag_status = pd.DataFrame(data['flags'])
    flag_status["FlagName"] = flag_status["FlagState"].map(flag_key).fillna("Unknown")
    df = pd.merge(df, flag_status, left_on = 'Lap', right_on = 'LapsCompleted', how = 'left')

    return df

# function to get lap data for a list of races
def get_lap_data(year, series_num, race_ids):
    lap_data = []
    # run the single race function above for each race id
    for race_id in race_ids: 
        race_data = fetch_single_race(year, series_num,race_id)
        race_data['race_id'] = race_id
        lap_data.append(race_data)

    lap_df = pd.concat(lap_data, ignore_index=True)
    return lap_df


# create a list of race ids
race_ids = races['race_id'].tolist()
series_num = 1

# get lap data from the function above
lap_df = get_lap_data(year, series_num, race_ids)


,Lap,LapTime,LapSpeed,RunningPos,Number,FullName,NASCARDriverID,LapsCompleted,FlagState,FlagName,race_id
0,0,NaN,None,18,24,William Byron,4184,0.0,8.0,Hot Track,5385
1,1,51.311,175.401,31,24,William Byron,4184,1.0,1.0,Green,5385
2,2,48.141,186.951,32,24,William Byron,4184,2.0,1.0,Green,5385
3,3,47.648,188.885,32,24,William Byron,4184,3.0,1.0,Green,5385
4,4,47.268,190.404,29,24,William Byron,4184,4.0,1.0,Green,5385
...,...,...,...,...,...,...,...,...,...,...,...
495,93,47.810,188.245,19,20,Christopher Bell,4153,93.0,1.0,Green,5385
496,94,47.505,189.454,21,20,Christopher Bell,4153,94.0,1.0,Green,5385
497,95,47.224,190.581,21,20,Christopher Bell,4153,95.0,1.0,Green,5385
498,96,47.081,191.160,21,20,Christopher Bell,4153,96.0,1.0,Green,5385


In [31]:


def fetch_single_result(year,series_num, race_id):
    race_url = f"https://cf.nascar.com/loopstats/prod/{year}/{series_num}/{race_id}.json"
    response = requests.get(race_url)
    data = response.json()
    # pull out lap data and create a row for each driver lap time
    df = pd.json_normalize(
        data,
        record_path='drivers',
        meta=['race_id']
    )
    df = df[['driver_id', 'closing_ps', 'laps', 'race_id']]

    return df

def get_result_data(year, series_num, race_ids):
    result_data = []
    # run the single race function above for each race id
    for race_id in race_ids: 
        race_data = fetch_single_result(year, series_num,race_id)
        race_data['race_id'] = race_id
        result_data.append(race_data)

    result_df = pd.concat(result_data, ignore_index=True)
    return result_df


results = get_result_data(year, series_num, race_ids)
results

,driver_id,closing_ps,laps,race_id
0,4184,7,200,5385
1,4045,14,200,5385
2,4153,26,200,5385
3,4096,29,200,5385
4,4025,21,200,5385
...,...,...,...,...
1351,4232,36,302,5428
1352,4070,37,302,5428
1353,4018,38,294,5428
1354,4272,39,247,5428


In [25]:
lap_df[(lap_df['race_id'] == 5385) & (lap_df['Number'] == '45')].head(201)

,Lap,LapTime,LapSpeed,RunningPos,Number,FullName,NASCARDriverID,LapsCompleted,FlagState,FlagName,race_id
5616,0,NaN,None,3,45,Tyler Reddick,4065,0.0,8.0,Hot Track,5385
5617,1,52.150,172.579,4,45,Tyler Reddick,4065,1.0,1.0,Green,5385
5618,2,47.961,187.652,4,45,Tyler Reddick,4065,2.0,1.0,Green,5385
5619,3,47.743,188.509,4,45,Tyler Reddick,4065,3.0,1.0,Green,5385
5620,4,47.785,188.344,4,45,Tyler Reddick,4065,4.0,1.0,Green,5385
...,...,...,...,...,...,...,...,...,...,...,...
5804,188,47.028,191.375,13,45,Tyler Reddick,4065,188.0,1.0,Green,5385
5805,189,46.916,191.832,14,45,Tyler Reddick,4065,189.0,1.0,Green,5385
5806,190,46.914,191.840,15,45,Tyler Reddick,4065,190.0,1.0,Green,5385
5807,191,46.941,191.730,16,45,Tyler Reddick,4065,191.0,1.0,Green,5385


In [32]:
merged_df = lap_df.merge(results, left_on=['race_id', 'NASCARDriverID'], right_on= ['race_id', 'driver_id'])

In [33]:
merged_df

,Lap,LapTime,LapSpeed,RunningPos,Number,FullName,NASCARDriverID,LapsCompleted,FlagState,FlagName,race_id,driver_id,closing_ps,laps
0,0,NaN,None,18,24,William Byron,4184,0.0,8.0,Hot Track,5385,4184,7,200
1,1,51.311,175.401,31,24,William Byron,4184,1.0,1.0,Green,5385,4184,7,200
2,2,48.141,186.951,32,24,William Byron,4184,2.0,1.0,Green,5385,4184,7,200
3,3,47.648,188.885,32,24,William Byron,4184,3.0,1.0,Green,5385,4184,7,200
4,4,47.268,190.404,29,24,William Byron,4184,4.0,1.0,Green,5385,4184,7,200
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
349334,245,28.698,125.444,30,71,Zane Smith #,4272,245.0,1.0,Green,5428,4272,39,247
349335,246,28.727,125.318,28,71,Zane Smith #,4272,246.0,1.0,Green,5428,4272,39,247
349336,247,28.644,125.681,28,71,Zane Smith #,4272,247.0,1.0,Green,5428,4272,39,247
349337,0,NaN,None,6,54,Ty Gibbs,4368,0.0,8.0,Hot Track,5428,4368,40,1
